# 13. CatBoost Regressor
### Handles Categorical Features

**CatBoost Regressor** offers state-of-the-art out-of-the-box accuracy for regression tasks containing both numerical and categorical features without manual target encoding.

#### Key Features:
- **Ordered Target Statistics**: Mitigates data leakage when encoding categories.
- **Symmetric / Oblivious Trees**: Accelerates CPU/GPU evaluation and prevents overfitting.
- High resilience to default hyperparameter settings.


In [ ]:
# Step 1: Install CatBoost if needed & Import Libraries
# !pip install catboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

try:
    from catboost import CatBoostRegressor
    print("CatBoost imported successfully.")
except ImportError:
    print("CatBoost not installed. Run: pip install catboost")


In [ ]:
# Step 2: Generate Regression Dataset with Categorical & Numerical Features
# Real estate price prediction scenario
np.random.seed(42)
n_samples = 600

df = pd.DataFrame({
    'sqft': np.random.normal(1800, 500, n_samples).round(),
    'bedrooms': np.random.choice([1, 2, 3, 4, 5], n_samples),
    'neighborhood': np.random.choice(['Downtown', 'Suburbs', 'Uptown', 'Rural'], n_samples),
    'property_type': np.random.choice(['Condo', 'Single Family', 'Townhouse'], n_samples)
})

# Price formulation with noise
price_base = (
    df['sqft'] * 150 +
    df['bedrooms'] * 15000 +
    (df['neighborhood'] == 'Downtown') * 80000 +
    (df['neighborhood'] == 'Uptown') * 40000 +
    (df['property_type'] == 'Single Family') * 30000 +
    np.random.normal(0, 20000, n_samples)
)
df['price'] = price_base

X = df.drop('price', axis=1)
y = df['price']
cat_features = ['neighborhood', 'property_type']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)


In [ ]:
# Step 3: Train CatBoost Regressor with Categorical Feature Support
try:
    cb_reg = CatBoostRegressor(
        iterations=150,
        learning_rate=0.08,
        depth=5,
        cat_features=cat_features,
        verbose=False,
        random_seed=42
    )
    cb_reg.fit(X_train, y_train, eval_set=(X_test, y_test))

    y_pred = cb_reg.predict(X_test)
    print(f"CatBoost Test R²: {r2_score(y_test, y_pred):.4f}")
    print(f"CatBoost Test RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
except NameError:
    print("CatBoost not available.")


In [ ]:
# Step 4: Feature Importance
try:
    imp = pd.Series(cb_reg.get_feature_importance(), index=X.columns).sort_values(ascending=False)
    plt.figure(figsize=(8, 4))
    imp.plot(kind='barh', color='indigo')
    plt.title("CatBoost Regressor: Feature Importance")
    plt.xlabel("Importance Score")
    plt.gca().invert_yaxis()
    plt.show()
except Exception as e:
    print(e)


### Summary
- **Strengths**: Best handling of mixed numerical and categorical features without leakage, superior out-of-the-box defaults.
- **Weaknesses**: Slower training than LightGBM on purely numerical datasets.
